# UMAP — manifold layout via a fuzzy neighbor graph

> Tutorial pair for [`umap.py`](umap.py).

## 1. Intuition
UMAP assumes the data lies on a manifold and builds a weighted nearest-neighbor
**graph** whose edge weights say "how surely are these two points neighbors?". It then
drops a low-dimensional graph and tugs it into shape: connected points **attract**,
random non-neighbors **repel**. Like t-SNE it shows clusters, but it tends to keep more
**global structure** and is fast.

## 2. Concept (the slide)
- **Fuzzy simplicial set:** a kNN graph where each edge carries a *membership
  strength* in $[0,1]$. Distances are measured from each point's nearest neighbor
  ($\rho_i$, local connectivity) and scaled by a per-point bandwidth $\sigma_i$.
- **Symmetrize** the directed memberships with a fuzzy **union** (probabilistic
  t-conorm).
- **Low-D layout:** edges get a smooth membership $\phi(d)=1/(1+a\,d^{2b})$.
  Minimize the fuzzy-set **cross-entropy** between high-D and low-D memberships —
  an **attractive** term on edges and a **repulsive** term on non-edges, optimized by
  SGD with **negative sampling**.

## 3. Math derivation

**Local fuzzy memberships.** For point $i$ with $k$ nearest neighbors, let
$\rho_i=\min_{j}d(\mathbf x_i,\mathbf x_j)$ (nearest-neighbor distance). Choose a
bandwidth $\sigma_i$ by binary search so the row's total membership is constant:

$$\sum_{j\in \mathrm{kNN}(i)}\exp\!\Big(-\frac{\max(0,\,d_{ij}-\rho_i)}{\sigma_i}\Big)=\log_2 k.$$

The directed membership strength is
$\;w_{j|i}=\exp\!\big(-\max(0,d_{ij}-\rho_i)/\sigma_i\big)$. Subtracting $\rho_i$
guarantees each point connects to its nearest neighbor with weight 1 (local
connectivity assumption).

**Symmetrization (fuzzy union / probabilistic t-conorm):**

$$w_{ij}=w_{j|i}+w_{i|j}-w_{j|i}\,w_{i|j}.$$

**Low-dimensional membership.** UMAP models the membership of a low-D edge of length
$d$ by the smooth curve

$$\phi(d)=\frac{1}{1+a\,d^{2b}},$$

with $(a,b)$ **fit by least squares** to the desired shape (controlled by `min_dist`
and `spread`): flat at 1 for $d\le\texttt{min\_dist}$, exponential decay beyond.

**Objective — fuzzy cross-entropy.** Treat each pair as a Bernoulli with high-D
"truth" $w_{ij}$ and low-D "prediction" $q_{ij}=\phi(\lVert\mathbf y_i-\mathbf y_j\rVert)$:

$$C=\sum_{i\ne j}\Big[\,w_{ij}\log\frac{w_{ij}}{q_{ij}}+(1-w_{ij})\log\frac{1-w_{ij}}{1-q_{ij}}\Big].$$

The first part is **attractive** (pull neighbors together where $w$ is large), the
second is **repulsive** (push non-neighbors apart where $w\approx0$).

**Gradients / forces.** Differentiating $C$ w.r.t. $\mathbf y_i$, with
$d^2=\lVert\mathbf y_i-\mathbf y_j\rVert^2$:

$$\text{attractive: } \frac{-2ab\,d^{2(b-1)}}{1+a\,d^{2b}}(\mathbf y_i-\mathbf y_j)\,w_{ij},
\qquad
\text{repulsive: } \frac{2b}{(\epsilon+d^2)(1+a\,d^{2b})}(\mathbf y_i-\mathbf y_j).$$

Evaluating the full repulsive sum is $O(n^2)$, so UMAP uses **negative sampling**:
for each edge, repel against a few random points. We also **decay the learning rate**
linearly over epochs. (Our NumPy version applies these same forces vectorized per
epoch for speed.)

## 4. NumPy implementation (fuzzy graph + attractive/repulsive SGD)

In [ ]:
# ===== actual implementation from umap.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _pairwise_sq_dists(X):
    """||x_i - x_j||^2 for all pairs."""
    sq = (X ** 2).sum(1)
    D = sq[:, None] - 2 * X @ X.T + sq[None, :]
    return np.maximum(D, 0)

def _fit_ab(min_dist=0.1, spread=1.0):
    r"""
    UMAP's low-dim membership is approximated by the smooth curve
        phi(d) = 1 / (1 + a * d^{2b}).
    We fit (a, b) by least squares so phi matches the target piecewise curve
        psi(d) = 1                       if d <= min_dist
                 exp(-(d - min_dist)/spread) otherwise.
    """
    from scipy.optimize import curve_fit

    xv = np.linspace(0, spread * 3, 300)
    yv = np.where(xv <= min_dist, 1.0, np.exp(-(xv - min_dist) / spread))

    def curve(x, a, b):
        return 1.0 / (1.0 + a * x ** (2 * b))

    (a, b), _ = curve_fit(curve, xv, yv, p0=(1.0, 1.0), maxfev=10000)
    return float(a), float(b)

import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def demo():
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    from sklearn.datasets import load_digits
    from sklearn.manifold import trustworthiness

    digits = load_digits()
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(digits.data), 300, replace=False)
    X, y = digits.data[idx], digits.target[idx]
    X = (X - X.mean(0)) / (X.std(0) + 1e-8)

    um = UMAPNumPy(n_components=2, n_neighbors=15, min_dist=0.1, n_epochs=120)
    Y = um.fit_transform(X)
    print(f"fitted membership curve: a={um.a:.3f}  b={um.b:.3f}")
    tw = trustworthiness(X, Y, n_neighbors=5)
    print(f"NumPy UMAP trustworthiness={tw:.3f}")

    # cluster structure: inter/intra class distance ratio in the map
    from itertools import combinations
    cen = np.stack([Y[y == c].mean(0) for c in np.unique(y)])
    intra = np.mean([np.linalg.norm(Y[i] - cen[y[i]]) for i in range(len(Y))])
    inter = np.mean([np.linalg.norm(cen[p] - cen[q]) for p, q in combinations(range(len(cen)), 2)])
    print(f"map intra={intra:.2f} inter={inter:.2f} ratio={inter/(intra+1e-9):.2f}")

    # Torch path uses autograd (heavier per step on CPU) -> smaller subset.
    Xs = X[:150]
    Yt = umap_torch(Xs, n_neighbors=15, n_iter=200)
    twt = trustworthiness(Xs, Yt, n_neighbors=5)
    print(f"Torch UMAP (n=150) trustworthiness={twt:.3f}")


class UMAPNumPy:
    r"""
    Step 1 — Fuzzy simplicial set (high-dim graph).
      For each point i, let rho_i be the distance to its nearest neighbor
      (local connectivity), and find sigma_i (a bandwidth) by binary search so
      that
          sum_{j in kNN(i)} exp(-(d_ij - rho_i)_+ / sigma_i) = log2(k).
      The directed membership strength is
          w_{j|i} = exp(-(d_ij - rho_i)_+ / sigma_i).
      Symmetrize with the probabilistic t-conorm (fuzzy union):
          w_{ij} = w_{j|i} + w_{i|j} - w_{j|i} w_{i|j}.

    Step 2 — Low-dim layout.
      Low-dim membership of an edge of length d is phi(d) = 1/(1 + a d^{2b}).
      Minimize the fuzzy cross-entropy between high-dim weights w_ij and
      low-dim memberships, which decomposes into an *attractive* term on graph
      edges and a *repulsive* term on (sampled) non-edges. We follow the
      gradients with SGD + negative sampling.
    """

    def __init__(self, n_components=2, n_neighbors=15, min_dist=0.1, spread=1.0,
                 n_epochs=200, lr=1.0, n_negative=5, seed=SEED):
        self.n_components = n_components
        self.n_neighbors = n_neighbors
        self.min_dist = min_dist
        self.spread = spread
        self.n_epochs = n_epochs
        self.lr = lr
        self.n_negative = n_negative
        self.seed = seed

    # --- fuzzy simplicial set -------------------------------------------
    def _fuzzy_graph(self, X):
        n = len(X)
        k = min(self.n_neighbors, n - 1)
        D = np.sqrt(_pairwise_sq_dists(X))
        # k nearest neighbors (exclude self at index 0 after sorting)
        knn_idx = np.argsort(D, axis=1)[:, 1:k + 1]
        knn_d = np.take_along_axis(D, knn_idx, axis=1)

        target = np.log2(k)
        W = np.zeros((n, n))
        for i in range(n):
            di = knn_d[i]
            rho = di[di > 0].min() if np.any(di > 0) else 0.0   # local connectivity
            # binary search sigma so the row sums to log2(k)
            lo, hi, sigma = 0.0, np.inf, 1.0
            for _ in range(64):
                psum = np.exp(-np.maximum(di - rho, 0) / sigma).sum()
                if abs(psum - target) < 1e-5:
                    break
                if psum > target:
                    hi = sigma
                    sigma = (lo + hi) / 2
                else:
                    lo = sigma
                    sigma = sigma * 2 if hi == np.inf else (lo + hi) / 2
            w = np.exp(-np.maximum(di - rho, 0) / sigma)
            W[i, knn_idx[i]] = w
        # symmetrize via probabilistic t-conorm (fuzzy union)
        W = W + W.T - W * W.T
        return W

    def fit_transform(self, X):
        X = np.asarray(X, float)
        rng = np.random.default_rng(self.seed)
        n = len(X)
        self.a, self.b = _fit_ab(self.min_dist, self.spread)

        W = self._fuzzy_graph(X)
        # edge list with weights (upper triangle is enough; graph is symmetric)
        ii, jj = np.where(W > 1e-3)
        mask = ii < jj
        ii, jj = ii[mask], jj[mask]
        wij = W[ii, jj]

        # spectral-ish init: a small random embedding (cheap, deterministic)
        Y = rng.normal(0, 10.0, (n, self.n_components))

        a, b = self.a, self.b
        n_edges = len(ii)
        # The per-edge / per-negative-sample loop is the textbook description,
        # but pure-Python loops are slow; we apply the *same* forces vectorized
        # per epoch (one batch of attractions over all edges, one batch of
        # repulsions over sampled non-edges) using scatter-add.
        for epoch in range(self.n_epochs):
            alpha = self.lr * (1.0 - epoch / self.n_epochs)   # LR decay

            # --- attractive forces on all graph edges at once ---
            diff = Y[ii] - Y[jj]                       # (n_edges, dim)
            d2 = (diff ** 2).sum(1) + 1e-12
            # d/dd2 of the attractive cross-entropy term:
            #   coef = -2 a b d2^{b-1} / (1 + a d2^b)
            coef = (-2.0 * a * b * d2 ** (b - 1.0)) / (1.0 + a * d2 ** b)
            grad = np.clip(coef[:, None] * diff, -4, 4) * wij[:, None]
            upd = alpha * grad
            np.add.at(Y, ii, upd)                      # endpoint i moves +grad
            np.add.at(Y, jj, -upd)                     # endpoint j moves -grad

            # --- repulsive forces against negative samples ---
            src = np.repeat(ii, self.n_negative)       # each edge tail repeated
            neg = rng.integers(0, n, size=len(src))    # random non-neighbors
            valid = neg != src
            src, neg = src[valid], neg[valid]
            diff = Y[src] - Y[neg]
            d2 = (diff ** 2).sum(1) + 1e-12
            # repulsive gradient coefficient
            coef = (2.0 * b) / ((1e-3 + d2) * (1.0 + a * d2 ** b))
            grad = np.clip(coef[:, None] * diff, -4, 4)
            np.add.at(Y, src, alpha * grad)
        self.embedding_ = Y
        self.graph_ = W
        return Y

## 5. PyTorch implementation (fuzzy cross-entropy via autograd)

In [ ]:
# ===== actual implementation from umap.py =====
def umap_torch(X, n_components=2, n_neighbors=15, min_dist=0.1, spread=1.0,
               n_iter=300, lr=1e-1, seed=SEED):
    r"""
    Optimize the fuzzy-set cross-entropy directly with autograd.

    With high-dim memberships w_ij (from the fuzzy graph) and low-dim
    memberships q_ij = 1/(1 + a ||y_i - y_j||^{2b}), the UMAP objective is the
    fuzzy cross-entropy
        CE = - sum_ij [ w_ij log q_ij + (1 - w_ij) log(1 - q_ij) ].
    We minimize a subsampled version (all edges attract, all pairs repel) with
    Adam — autograd handles the messy gradient.
    """
    dev = get_device()
    X = np.asarray(X, float)
    n = len(X)
    a, b = _fit_ab(min_dist, spread)

    builder = UMAPNumPy(n_components, n_neighbors, min_dist, spread, seed=seed)
    W = builder._fuzzy_graph(X)
    Wt = torch.as_tensor(W, dtype=torch.float32, device=dev)
    eye = torch.eye(n, device=dev, dtype=torch.bool)

    g = torch.Generator(device="cpu").manual_seed(seed)
    Y = (torch.randn(n, n_components, generator=g) * 10.0).to(dev).requires_grad_(True)
    opt = torch.optim.Adam([Y], lr=lr)

    for _ in range(n_iter):
        # algebraic squared distances (faster than cdist under autograd on CPU)
        sq = (Y ** 2).sum(1)
        d2 = (sq[:, None] - 2.0 * (Y @ Y.T) + sq[None, :]).clamp_min(0.0)
        q = 1.0 / (1.0 + a * d2.clamp_min(1e-9) ** b)
        q = q.masked_fill(eye, 0.0).clamp(1e-6, 1 - 1e-6)
        ce = -(Wt * q.log() + (1.0 - Wt) * (1.0 - q).log())
        ce = ce.masked_fill(eye, 0.0)
        loss = ce.sum() / (n * n)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return Y.detach().cpu().numpy()

## 6. Train / run — fitted (a,b), trustworthiness, cluster ratio

In [ ]:
demo()

## 7. Visualization — digits laid out by UMAP

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import umap as M

from sklearn.datasets import load_digits
digits = load_digits()
rng = np.random.default_rng(0)
idx = rng.choice(len(digits.data), 300, replace=False)
X, y = digits.data[idx], digits.target[idx]
X = (X - X.mean(0)) / (X.std(0) + 1e-8)

Y = M.UMAPNumPy(n_components=2, n_neighbors=15, min_dist=0.1, n_epochs=120).fit_transform(X)

plt.figure(figsize=(6, 5))
sc = plt.scatter(Y[:, 0], Y[:, 1], c=y, cmap="tab10", s=18)
plt.colorbar(sc, label="digit"); plt.title("UMAP of handwritten digits")
plt.xlabel("dim 1"); plt.ylabel("dim 2"); plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **`n_neighbors`** trades local vs global structure (small → local detail, large →
  global shape); **`min_dist`** controls how tightly points clump.
- UMAP usually preserves **global structure** better than t-SNE and is faster
  (negative sampling), and unlike t-SNE it can **embed new points**.
- Still a **visualization** tool: inter-cluster distances and densities are not
  literal — don't read absolute geometry into the map.
- The layout is **stochastic / non-convex**; fix the seed for reproducibility.
- This is a **simplified** UMAP (faithful fuzzy graph + force layout); the real library
  adds approximate kNN, a spectral initialization, and a sampled edge schedule.